# IndustrialGuard AI ? Dataset exploration

Read-only inspection of original archive members. Nothing is extracted. MPDD is the primary mechanical-component benchmark; MVTec AD is the complementary broader industrial anomaly benchmark.

In [ ]:
from pathlib import Path
import csv, io, json, sys
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from industrialguard.data.archive_reader import ArchiveReader
from industrialguard.data.dataset_audit import read_config
REPORTS = ROOT / 'reports' / 'dataset_audit'
CONFIG = read_config(ROOT / 'configs' / 'dataset.yaml')
REPORTS.mkdir(parents=True, exist_ok=True)


## Dataset overview and category distribution

Counts come from inventories built from actual archive member paths.

In [ ]:
summaries = {name: json.loads((REPORTS / f'{name}_summary.json').read_text()) for name in CONFIG['datasets']}
for name, summary in summaries.items():
    print(name, {key: summary[key] for key in ('image_count', 'normal_images', 'anomalous_images', 'ground_truth_mask_members', 'missing_masks')})
    for category, counts in summary['by_category'].items():
        ratio = counts['normal'] / max(counts['anomalous'], 1)
        print(f"  {category}: normal={counts['normal']} anomalous={counts['anomalous']} normal/anomalous={ratio:.2f}")
fig, axes = plt.subplots(1, len(summaries), figsize=(15, 6))
if len(summaries) == 1: axes = [axes]
for ax, (name, summary) in zip(axes, summaries.items()):
    categories = list(summary['by_category'])
    normal = [summary['by_category'][category]['normal'] for category in categories]
    anomalous = [summary['by_category'][category]['anomalous'] for category in categories]
    ax.barh(categories, normal, label='Normal')
    ax.barh(categories, anomalous, left=normal, label='Anomalous')
    ax.set_title(f'{name}: per-category samples'); ax.set_xlabel('Images'); ax.legend()
fig.tight_layout(); fig.savefig(REPORTS / 'category_distribution.png', dpi=150); plt.show()


## Normal, anomalous, and defect type distributions

In [ ]:
fig, axes = plt.subplots(1, len(summaries), figsize=(14, 5))
if len(summaries) == 1: axes = [axes]
for ax, (name, summary) in zip(axes, summaries.items()):
    ax.bar(['Normal', 'Anomalous'], [summary['normal_images'], summary['anomalous_images']], color=['#4c78a8', '#f58518'])
    ax.set_title(name); ax.set_ylabel('Images')
fig.tight_layout(); fig.savefig(REPORTS / 'normal_anomalous_distribution.png', dpi=150); plt.show()
for name in summaries:
    with (REPORTS / f'{name}_inventory.csv').open(newline='', encoding='utf-8') as stream:
        rows = list(csv.DictReader(stream))
    defect_counts = Counter(row['defect_type'] for row in rows if row['defect_type'])
    print(name, 'defect types:', dict(defect_counts))


## Resolution and mask availability

Resolution data excludes masks and includes successfully decoded image records.

In [ ]:
for name, summary in summaries.items():
    resolutions = summary['image_resolutions']
    plt.figure(figsize=(8, 3)); plt.bar(resolutions.keys(), resolutions.values()); plt.xticks(rotation=45, ha='right')
    plt.title(f'{name}: image resolutions'); plt.ylabel('Images'); plt.tight_layout()
    plt.savefig(REPORTS / f'{name}_resolution_distribution.png', dpi=150); plt.show()
    print(name, 'matched masks:', summary['matched_masks'], 'missing:', summary['missing_masks'], 'mask members:', summary['ground_truth_mask_members'])


## Read image and mask members directly, then visualize overlays

The iterator decompresses sequentially and retains only selected members.

In [ ]:
def load_selected(reader, wanted):
    wanted = set(wanted); found = {}
    for name, payload in reader.iter_member_bytes():
        if name in wanted: found[name] = payload
        if found.keys() >= wanted: break
    if found.keys() != wanted:
        raise FileNotFoundError(f"Archive members not found: {sorted(wanted - found.keys())}")
    return found
examples = {}
for name, spec in CONFIG['datasets'].items():
    with (REPORTS / f'{name}_inventory.csv').open(newline='', encoding='utf-8') as stream:
        rows = list(csv.DictReader(stream))
    normal_row = next((item for item in rows if item['image_type'] == 'normal'), None)
    anomaly_row = next((item for item in rows if item['image_type'] == 'anomalous' and item['mask_path']), None)
    if normal_row is None or anomaly_row is None:
        print(name, 'does not have both normal and matched anomalous examples'); continue
    reader = ArchiveReader(spec['archive'], spec['format'])
    payloads = load_selected(reader, [normal_row['image_path'], anomaly_row['image_path'], anomaly_row['mask_path']])
    normal_image = Image.open(io.BytesIO(payloads[normal_row['image_path']])).convert('RGB')
    anomaly_image = Image.open(io.BytesIO(payloads[anomaly_row['image_path']])).convert('RGB')
    mask = Image.open(io.BytesIO(payloads[anomaly_row['mask_path']])).convert('L')
    examples[name] = (normal_image, anomaly_image, mask, anomaly_row)
fig, axes = plt.subplots(len(examples), 4, figsize=(15, 4 * len(examples)))
if len(examples) == 1: axes = [axes]
for axrow, (name, (normal_image, anomaly_image, mask, row)) in zip(axes, examples.items()):
    overlay = anomaly_image.copy(); overlay_mask = mask.resize(anomaly_image.size, Image.Resampling.NEAREST)
    red = Image.new('RGB', anomaly_image.size, (255, 0, 0)); overlay.paste(red, mask=overlay_mask)
    axrow[0].imshow(normal_image); axrow[0].set_title(f'{name}: normal example')
    axrow[1].imshow(anomaly_image); axrow[1].set_title(f"{row['category']} / {row['defect_type']}")
    axrow[2].imshow(mask, cmap='gray'); axrow[2].set_title('Ground-truth mask')
    axrow[3].imshow(overlay); axrow[3].set_title('Overlay')
    for ax in axrow: ax.axis('off')
fig.tight_layout(); fig.savefig(REPORTS / 'image_mask_overlay_examples.png', dpi=150); plt.show()


## Quality review

Errors and warnings are separate in `validation_report.json`. Review missing annotations, invalid images, empty masks, dimension mismatches, duplicate archive paths, and unexpected directory structures before training.

In [ ]:
quality = json.loads((REPORTS / 'validation_report.json').read_text())
for name, report in quality.items():
    print(name, 'errors:', len(report['errors']), 'warnings:', len(report['warnings']), report['counts'])
